# AI Agronomist - Crop Disease Classifier Training 🌿

This notebook contains the complete pipeline for training a deep learning model to identify diseases in Cassava and Maize crops. We use **Transfer Learning** with the **MobileNetV2** architecture for high efficiency on mobile devices.

---

### Step 1: Import Dependencies
We need TensorFlow for the AI, Pandas for data handling, and Matplotlib for visualizing our accuracy.

In [ ]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

# Ensure plots appear inside the notebook
%matplotlib inline
print("TensorFlow version:", tf.__version__)

### Step 2: Data Configuration
Set the paths to your training images and the CSV file that maps images to labels.

In [ ]:
DATA_DIR = r"c:\Users\colli\OneDrive\Desktop\Angel\train_images"
CSV_PATH = r"c:\Users\colli\OneDrive\Desktop\Angel\train.csv"

# Target images size for MobileNetV2
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

### Step 3: Data Augmentation & Loading
We use `ImageDataGenerator` to "artificially" increase our dataset by flipping and rotating images. This prevents the model from just memorizing the pictures.

In [ ]:
df = pd.read_csv(CSV_PATH)
df['label'] = df['label'].astype(str) # Keras needs strings for categorical labels

datagen = ImageDataGenerator(
    rescale=1./255, 
    validation_split=0.2, 
    horizontal_flip=True, 
    rotation_range=20
)

print("Loading Training Data...")
train_gen = datagen.flow_from_dataframe(
    df, directory=DATA_DIR, x_col='image_id', y_col='label', 
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, subset='training'
)

print("Loading Validation Data...")
val_gen = datagen.flow_from_dataframe(
    df, directory=DATA_DIR, x_col='image_id', y_col='label', 
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, subset='validation'
)

### Step 4: Building the Neural Network
We use **Transfer Learning**: We take a pre-trained model (MobileNetV2) and attach our own "head" (the classifier) to identify our specific crop diseases.

In [ ]:
print("Building MobileNetV2 Architecture...")
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Freeze the pre-trained weights

x = GlobalAveragePooling2D()(base_model.output)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x) # Helps prevent overfitting
predictions = Dense(5, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

### Step 5: Training
This is where the "learning" happens. The model will go through the images and update its weights to minimize errors.

In [ ]:
EPOCHS = 5 # Start with 5 epochs for the demo

print("Starting Training Session...")
history = model.fit(
    train_gen, 
    epochs=EPOCHS, 
    validation_data=val_gen
)

### Step 6: Visualizing Performance
Compare the training accuracy vs validation accuracy. If they are close, the model is generalizing well!

In [ ]:
plt.figure(figsize=(12, 5))

# Plot Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy Scores')
plt.legend()

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss Scores')
plt.legend()

plt.show()

### Step 7: Saving the Model
Save the trained weights to a file so the Streamlit app can use it.

In [ ]:
if not os.path.exists('models'): os.makedirs('models')
model.save('models/cassava_model_v2.h5')
print("Success! Model saved to models/cassava_model_v2.h5")